In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# =============================
# 1. Load all CSVs
# =============================
# NOTE: Data paths are omitted in the public version due to privacy constraints.
path = "<PRIVATE_DATA_PATH>/"

basic_info = pd.read_csv(path + "basic_info.csv")
height_weight = pd.read_csv(path + "height_weight.csv")
labs = pd.read_csv(path + "labs.csv")
med = pd.read_csv(path + "med.csv")
notes = pd.read_csv(path + "notes.csv")
mortality = pd.read_excel(path + "r26_mortality_data.xlsx")
selected_disease = pd.read_csv(path + "selected_disease.csv")
social_hist = pd.read_csv(path + "social_hist.csv")
surgery = pd.read_csv(path + "surgery.csv")

# **Disease table processing**

In [2]:
selected_disease.columns

Index(['mrn', 'pat_id', 'pat_enc_csn_id', 'dx_type', 'dx_id', 'dx_name',
       'icd10', 'entry_date', 'resolved_date', 'status_comment'],
      dtype='object')

In [ ]:
selected_disease.head()

In [ ]:
import pandas as pd
import numpy as np

df = selected_disease.copy()
df['entry_date'] = pd.to_datetime(df['entry_date'], errors='coerce')
df['resolved_date'] = pd.to_datetime(df['resolved_date'], errors='coerce')
df['dx_name_clean'] = df['dx_name'].fillna('').str.lower()
df['icd10_clean'] = df['icd10'].fillna('').astype(str).str.upper()

# Identify AP / CP
df['is_ap'] = df['dx_name_clean'].str.contains('acute pancreatitis', na=False) | df['icd10_clean'].str.startswith('K85')
df['is_cp'] = df['dx_name_clean'].str.contains('chronic pancreatitis', na=False) | df['icd10_clean'].str.startswith('K86.1')

# Aggregate AP and CP dates
ap_dates = (
    df[df['is_ap'] & df['entry_date'].notna()]
    .sort_values(['pat_id', 'entry_date'])
    .groupby('pat_id', as_index=False)['entry_date']
    .agg(set)
    .rename(columns={'entry_date': 'ap_date_set'})
)

cp_dates = (
    df[df['is_cp'] & df['entry_date'].notna()]
    .sort_values(['pat_id', 'entry_date'])
    .groupby('pat_id', as_index=False)['entry_date']
    .first()
    .rename(columns={'entry_date': 'cp_date'})
)

# Merge to patient level
patients = pd.DataFrame({'pat_id': pd.unique(df['pat_id'])})
patients = patients.merge(ap_dates, on='pat_id', how='left')
patients = patients.merge(cp_dates, on='pat_id', how='left')

def earliest_date(x):
    if isinstance(x, (set, list)):
        return min(pd.to_datetime(list(x), errors='coerce'))
    return pd.to_datetime(x, errors='coerce')

patients['earliest_ap_date'] = patients['ap_date_set'].apply(earliest_date)

# Compute timing
patients['cp_date'] = pd.to_datetime(patients['cp_date'], errors='coerce')

patients['days_ap_to_cp'] = (
    patients['cp_date'] - patients['earliest_ap_date']
).dt.days

# -------------------------------------------------------------------
# Drop conditions
# -------------------------------------------------------------------
# 1. Drop patients with no AP at all
patients = patients[patients['earliest_ap_date'].notna()].copy()

# 2. Drop CP before AP (prevalent CP)
patients = patients[~(patients['days_ap_to_cp'] < 0)].copy()

# 3. Drop CP within < 6 months ( < 180 days )
patients = patients[~((patients['days_ap_to_cp'] >= 0) & (patients['days_ap_to_cp'] < 365.25 / 2))].copy()

# 4. Drop only one AP observation with no CP later
patients["ap_count"] = patients["ap_date_set"].apply(
    lambda x: len(x) if isinstance(x, (set, list)) else 0
)
to_drop = (patients["ap_count"] == 1) & (patients["cp_date"].isna())
patients = patients[~to_drop].copy()

# Create label:
#   0 = AP with NO CP
#   1 = AP with CP at least 6 months later
patients['cp_label'] = np.where(
    patients['cp_date'].notna(),
    1,   # CP exists AND passed all filters above (≥6 months)
    0    # no CP
)

# For modeling, time-to-event only valid in CP group
patients['time_to_cp_days'] = np.where(
    patients['cp_label'] == 1,
    patients['days_ap_to_cp'],
    np.nan
)

print("Distribution:")
print(patients['cp_label'].value_counts(dropna=False))
print(patients.head())


In [5]:
def last_ap_before_cp(ap_dates, cp_date):
    """
    ap_dates: set / list / scalar of AP dates
    cp_date: scalar CP date or NaN

    returns: pandas.Timestamp or NaT
    """
    if ap_dates is None or (isinstance(ap_dates, float) and pd.isna(ap_dates)):
        return pd.NaT

    # normalize AP dates to datetime
    if isinstance(ap_dates, (set, list)):
        ap_dt = pd.to_datetime(list(ap_dates), errors="coerce")
    else:
        ap_dt = pd.to_datetime([ap_dates], errors="coerce")

    ap_dt = ap_dt.dropna()

    if len(ap_dt) == 0:
        return pd.NaT

    cp_dt = pd.to_datetime(cp_date, errors="coerce")

    # If CP exists → only APs before CP
    if not pd.isna(cp_dt):
        ap_dt = ap_dt[ap_dt < cp_dt]

    return ap_dt.max() if len(ap_dt) > 0 else pd.NaT
patients["last_ap_date"] = patients.apply(
    lambda r: last_ap_before_cp(r["ap_date_set"], r["cp_date"]),
    axis=1
)

In [ ]:
print(patients)

# **Basic**

In [7]:
df = basic_info.copy()

In [8]:
df.columns

Index(['mrn', 'pat_id', 'mrn.1', 'race', 'ethnicity', 'sex', 'birth_date',
       'death_date'],
      dtype='object')

In [ ]:
def process_basic_info(df, patients):
    df = df.copy()
    
    # Sex encoding
    df['sex'] = df['sex'].map({'Male':1, 'Female':0, 'X':2})
    
    # Alive status
    df['death_date'] = pd.to_datetime(df['death_date'], errors='coerce')
    df['alive'] = df['death_date'].isna().astype(int)
    
    # Age computation
    df['birth_date'] = pd.to_datetime(df['birth_date'], errors='coerce')
    today = pd.Timestamp('today')

    df.loc[df['alive'] == 1, 'age'] = (
        (today - df['birth_date']).dt.days / 365.25
    )
    df.loc[df['alive'] == 0, 'age'] = (
        (df['death_date'] - df['birth_date']).dt.days / 365.25
    )
    
    # Select final columns
    cols = ['pat_id', 'age', 'race', 'ethnicity', 'sex', 'alive']
    df = df[cols]

    # ⭐ INNER MERGE with final patients table
    df = patients.merge(df, on='pat_id', how='inner')

    return df

basic_info_feat = process_basic_info(basic_info, patients)
basic_info_feat


# **Height/Weight**

In [ ]:
height_weight

In [11]:
df = height_weight.copy()

def height_to_cm(h):
    try:
        if "'" in h:
            feet, inches = h.split("'")
            inches = inches.replace('"','')
            return (int(feet)*12 + float(inches))*2.54
        else:
            return float(h)  # already numeric
    except:
        return np.nan

def weight_to_kg(w):
    try:
        w = float(w)  # convert string to float
    except (ValueError, TypeError):
        return np.nan
    return w/10 * 0.453592


def avg_weight_height(df, patients):
    df = df.copy()
    print("=== START ===")
    print("rows in raw df:", df.shape[0])
    print("unique pats in raw df:", df['pat_id'].nunique())
    print("unique patients table:", patients['pat_id'].nunique())

    # -------------------------------
    # 1) Attach CP timing information
    # -------------------------------
    patients_sub = patients[['pat_id', 'cp_label', 'cp_date', 'earliest_ap_date','last_ap_date']].copy()
    patients_sub['window_end'] = patients_sub['earliest_ap_date'] + pd.to_timedelta(365.25 * 10, unit='D')

    df = df.merge(patients_sub, on='pat_id', how='inner')
    print("\nAfter merge with patients_sub:")
    print("rows:", df.shape[0])
    print("unique pats:", df['pat_id'].nunique())
    print("cp_label value_counts:\n", df['cp_label'].value_counts(dropna=False))

    # Convert df.contact_date
    df['contact_date'] = pd.to_datetime(df['contact_date'], errors='coerce')
    print("\ncontact_date NaN fraction:", df['contact_date'].isna().mean())

    # -------------------------------
    # 2) Convert height / weight
    # -------------------------------
    df['weight_kg'] = df.apply(
        lambda x: weight_to_kg(x['observation_value']) if x['name'] == 'WEIGHT' else np.nan,
        axis=1
    )
    df['height_cm'] = df.apply(
        lambda x: height_to_cm(x['observation_value']) if x['name'] == 'HEIGHT' else np.nan,
        axis=1
    )

    print("\nAfter conversion:")
    print("non-NaN weight_kg rows:", df['weight_kg'].notna().sum())
    print("non-NaN height_cm rows:", df['height_cm'].notna().sum())
    print("example names:", df['name'].value_counts().head())

    # Remove invalid data
    before_valid = df.shape[0]
    df = df[
        (df['weight_kg'].isna() | ((df['weight_kg'] >= 30) & (df['weight_kg'] <= 250))) &
        (df['height_cm'].isna() | ((df['height_cm'] >= 120) & (df['height_cm'] <= 220)))
    ]
    print("\nAfter validity filter:")
    print("rows:", df.shape[0], " (dropped:", before_valid - df.shape[0], ")")
    print("non-NaN weight_kg:", df['weight_kg'].notna().sum())
    print("non-NaN height_cm:", df['height_cm'].notna().sum())

    # -------------------------------
    # 3) Apply time filtering rules
    # -------------------------------
    # CP patients → keep only contact_date ≤ cp_date
    mask_cp = (df['cp_label'] == 1) & (df['contact_date'] < df['cp_date'])

    # Non-CP patients → keep only contact_date ≤ last_ap_date
    mask_censor = (df['cp_label'] == 0) & (df['contact_date'] <= df['last_ap_date'])
    # Global cap: within window_end for everyone

    before_time = df.shape[0]

    df = df[(mask_cp| mask_censor )].copy()

    print("\nAfter time filtering:")
    print("rows:", df.shape[0], " (dropped:", before_time - df.shape[0], ")")
    print("non-NaN weight_kg:", df['weight_kg'].notna().sum())
    print("non-NaN height_cm:", df['height_cm'].notna().sum())
    print("patients with any row after time filter:", df['pat_id'].nunique())


    # -------------------------------
    # 4) Compute average weight/height
    # -------------------------------
    weight_avg = (
        df.dropna(subset=['weight_kg'])
          .groupby('pat_id')['weight_kg']
          .mean()
          .reset_index()
    )
    height_avg = (
        df.dropna(subset=['height_cm'])
          .groupby('pat_id')['height_cm']
          .mean()
          .reset_index()
    )

    print("\nPatients with weight_avg:", weight_avg['pat_id'].nunique())
    print("Patients with height_avg:", height_avg['pat_id'].nunique())

    patient_hw = pd.merge(weight_avg, height_avg, on='pat_id', how='outer')
    print("Patients with any h/w avg:", patient_hw['pat_id'].nunique())

    # -------------------------------
    # ⭐ 5) INCLUDE all patients (even with no h/w)
    # -------------------------------
    patient_hw_full = patients[['pat_id']].merge(patient_hw, on='pat_id', how='left')
    print("\nFinal patient_hw_full shape:", patient_hw_full.shape)
    print("non-NaN weight_kg pats:", patient_hw_full['weight_kg'].notna().sum())
    print("non-NaN height_cm pats:", patient_hw_full['height_cm'].notna().sum())
    print("=== END ===\n")

    return patient_hw_full


weight_height = avg_weight_height(df, patients)


=== START ===
rows in raw df: 1353290
unique pats in raw df: 25784
unique patients table: 4904

After merge with patients_sub:
rows: 293274
unique pats: 4869
cp_label value_counts:
 cp_label
0    235515
1     57759
Name: count, dtype: int64

contact_date NaN fraction: 0.0

After conversion:
non-NaN weight_kg rows: 158903
non-NaN height_cm rows: 134371
example names: name
WEIGHT    158903
HEIGHT    134371
Name: count, dtype: int64

After validity filter:
rows: 292409  (dropped: 865 )
non-NaN weight_kg: 158058
non-NaN height_cm: 134351

After time filtering:
rows: 193343  (dropped: 99066 )
non-NaN weight_kg: 104144
non-NaN height_cm: 89199
patients with any row after time filter: 4798

Patients with weight_avg: 4784
Patients with height_avg: 4754
Patients with any h/w avg: 4798

Final patient_hw_full shape: (4904, 3)
non-NaN weight_kg pats: 4784
non-NaN height_cm pats: 4754
=== END ===



In [ ]:
weight_height


# **Labs**

In [13]:
labs.columns

Index(['mrn', 'order_proc_id', 'line', 'pat_id', 'pat_enc_csn_id', 'proc_id',
       'order_description', 'ordering_date', 'result_time', 'component_id',
       'component_name', 'numeric_value', 'raw_value', 'reference_unit',
       'result_flag', 'result_status', 'lab_status', 'loinc_code'],
      dtype='object')

In [ ]:
labs.head()

In [15]:
def compute_lab_features(labs, patients, relevant_labs):
    labs = labs.copy()

    # -------------------------------
    # 1) Attach CP timing information
    # -------------------------------
    patients_sub = patients[['pat_id', 'cp_label', 'cp_date',
                             'earliest_ap_date', 'last_ap_date']].copy()
    patients_sub['window_end'] = patients_sub['earliest_ap_date'] + pd.to_timedelta(365.25 * 10, unit='D')

    labs = labs.merge(patients_sub, on='pat_id', how='inner')

    # -------------------------------
    # 2) Filter to relevant labs
    # -------------------------------
    labs = labs[labs['component_name'].isin(relevant_labs)]

    # Convert timestamp
    labs['result_time'] = pd.to_datetime(labs['result_time'], errors='coerce')

    # -------------------------------
    # 3) Apply time filtering rules
    # -------------------------------
    # CP patients → keep only result_time ≤ cp_date
    mask_cp = (labs['cp_label'] == 1) & (labs['result_time'] < labs['cp_date'])

    # Non-CP patients → keep only result_time ≤ last_ap_date
    mask_censor = (labs['cp_label'] == 0) & (labs['result_time'] <= labs['last_ap_date'])

    labs = labs[(mask_cp | mask_censor)].copy()

    # -------------------------------
    # 4) Compute per-patient mean of each lab
    # -------------------------------
    labs_mean = (
        labs.groupby(['pat_id', 'component_name'])['numeric_value']
            .mean()
            .reset_index()
            .pivot(index='pat_id', columns='component_name', values='numeric_value')
            .reset_index()
    )

    # -------------------------------
    # 5) INCLUDE ALL PATIENTS (even no labs)
    # -------------------------------
    labs_mean_full = patients[['pat_id']].merge(labs_mean, on='pat_id', how='left')

    return labs_mean_full

In [16]:
relevant_labs = ['GLUCOSE', 'CREATININE', 'AMYLASE', 'LIPASE']
labs_mean = compute_lab_features(labs, patients, relevant_labs)

In [ ]:
labs_mean

# **Med**

In [18]:
med.columns

Index(['mrn', 'order_med_id', 'pat_id', 'pat_enc_csn_id', 'ordering_time',
       'order_start_time', 'order_end_time', 'discontinue_time', 'sig',
       'discrete_dose', 'dose_unit', 'lastdose', 'frequency', 'route',
       'medication_id', 'medication_name', 'generic_name', 'thera_class_c',
       'thera_class', 'pharm_class_c', 'pharm_class', 'pharm_subclass_c',
       'pharm_subclass', 'order_class', 'order_status', 'ordering_mode'],
      dtype='object')

In [ ]:
med.head()

In [20]:
def compute_pancreatic_enzyme_features(med, patients):
    med = med.copy()

    # -------------------------------
    # 1) Filter to relevant meds
    # -------------------------------
    relevant_med = med[
        med['generic_name'].str.contains('lipase|amylase|protease', case=False, na=False)
    ].copy()

    relevant_med['order_start_time'] = pd.to_datetime(
        relevant_med['order_start_time'], errors='coerce'
    )
    relevant_med['order_end_time'] = pd.to_datetime(
        relevant_med['order_end_time'], errors='coerce'
    )

    # -------------------------------
    # 2) Attach patient timing info
    # -------------------------------
    patients_sub = patients[['pat_id', 'cp_label', 'cp_date',
                             'earliest_ap_date', 'last_ap_date']].copy()
    patients_sub['window_end'] = patients_sub['earliest_ap_date'] + pd.to_timedelta(365.25 * 10, unit='D')

    relevant_med = relevant_med.merge(patients_sub, on='pat_id', how='inner')

    # -------------------------------
    # 3) Apply time filtering rules
    # -------------------------------
    # Use order_start_time as the exposure time stamp
    t = relevant_med['order_start_time']

    # CP patients → keep only meds with start ≤ cp_date
    mask_cp = (relevant_med['cp_label'] == 1) & (t < relevant_med['cp_date'])

    # Non-CP patients → keep only meds with start ≤ last_ap_date
    mask_censor = (relevant_med['cp_label'] == 0) & (t <= relevant_med['last_ap_date'])

    relevant_med = relevant_med[(mask_cp | mask_censor)].copy()

    # -------------------------------
    # 4) Aggregate per patient
    # -------------------------------
    med_agg = (
        relevant_med.groupby('pat_id')
        .agg(
            ever_taken_pancreatic_enzymes=('order_med_id', lambda x: 1),
            num_prescriptions=('order_med_id', 'count'),
            first_med_start=('order_start_time', 'min'),
            last_med_end=('order_end_time', 'max'),
        )
        .reset_index()
    )

    # -------------------------------
    # 5) INCLUDE all patients
    # -------------------------------
    med_agg_full = patients[['pat_id']].merge(med_agg, on='pat_id', how='left')

    # Optional: make the indicator explicit 0/1
    med_agg_full['ever_taken_pancreatic_enzymes'] = (
        med_agg_full['ever_taken_pancreatic_enzymes'].fillna(0).astype(int)
    )

    return med_agg_full
med_features = compute_pancreatic_enzyme_features(med, patients)

In [ ]:
med_features[med_features["ever_taken_pancreatic_enzymes"] == 1]

# **Surgery**

In [22]:
surgery.columns

Index(['mrn', 'pat_id', 'pat_enc_csn_id', 'surgery_date', 'proc_id',
       'procedure_name', 'service', 'site', 'laterality', 'anesthesia_type',
       'anesthesia_start', 'anesthesia_stop', 'wound_class', 'scheduled',
       'performed', 'panel'],
      dtype='object')

In [ ]:
surgery.head()

In [24]:
def compute_surgery_features(surgery, patients):
    surgery_feat = surgery.copy()

    # -------------------------------
    # 1) Convert YYYYMMDD float → datetime
    # -------------------------------
    surgery_feat['surgery_date'] = surgery_feat['surgery_date'].apply(
        lambda x: pd.to_datetime(str(int(x)), format='%Y%m%d', errors='coerce')
        if pd.notna(x) else pd.NaT
    )

    # Keep only valid procedure names
    surgery_filtered = surgery_feat[surgery_feat['procedure_name'].notna()].copy()

    # -------------------------------
    # 2) Attach patient timing info
    # -------------------------------
    patients_sub = patients[['pat_id', 'cp_label', 'cp_date',
                             'earliest_ap_date', 'last_ap_date']].copy()
    patients_sub['window_end'] = patients_sub['earliest_ap_date'] + pd.to_timedelta(365.25 * 10, unit='D')

    surgery_filtered = surgery_filtered.merge(patients_sub, on='pat_id', how='inner')

    # -------------------------------
    # 3) Apply time filtering rules
    # -------------------------------
    t = surgery_filtered['surgery_date']

    # CP patients → surgery_date ≤ cp_date
    mask_cp = (surgery_filtered['cp_label'] == 1) & (t < surgery_filtered['cp_date'])

    # Non-CP patients → surgery_date ≤ last_ap_date
    mask_censor = (surgery_filtered['cp_label'] == 0) & (t <= surgery_filtered['last_ap_date'])
    
    surgery_filtered = surgery_filtered[(mask_cp | mask_censor)].copy()

    # -------------------------------
    # 4) Aggregate per patient
    # -------------------------------
    surgery_agg = (
        surgery_filtered[surgery_filtered["performed"] == 1]
        .groupby("pat_id")
        .agg(
            ever_had_cholecystectomy=('procedure_name', lambda x: 1),
            num_cholecystectomies=('procedure_name', 'count'),
            first_surgery_date=('surgery_date', 'min'),
            last_surgery_date=('surgery_date', 'max'),
        )
        .reset_index()
    )

    # -------------------------------
    # 5) INCLUDE all patients
    # -------------------------------
    surgery_agg_full = patients[['pat_id']].merge(surgery_agg, on='pat_id', how='left')

    # Make the indicator explicit 0/1
    surgery_agg_full['ever_had_cholecystectomy'] = (
        surgery_agg_full['ever_had_cholecystectomy'].fillna(0).astype(int)
    )

    return surgery_agg_full
surgery_features = compute_surgery_features(surgery, patients)

In [ ]:
surgery_features

# **Social History**

In [26]:
social_hist_feat =  social_hist.copy()

In [27]:
social_hist_feat.columns

Index(['mrn', 'pat_id', 'tobacco_use', 'smokeless_tobacco_use', 'alcohol_use',
       'iv_drug_use', 'sexually_active', 'contact_date'],
      dtype='object')

In [ ]:
social_hist_feat.head()

In [29]:
social_hist_feat['sexually_active'].value_counts(dropna = False)

sexually_active
NaN              263933
Yes              210129
Not Currently    175361
Not Asked        118466
Never             60169
Name: count, dtype: int64

In [ ]:
def clean_social_history(df):
    df = df.copy()
    
    # --- Tobacco ---
    df['tobacco_use_clean'] = df['tobacco_use'].replace({
        'Never': 'Never', 'Passive Smoke Exposure - Never Smoker': 'Never',
        'Former': 'Former',
        'Every Day': 'Current', 'Light Smoker': 'Current',
        'Some Days': 'Current', 'Smoker, Current Status Unknown': 'Unknown', 'Heavy Smoker': 'Current',
        'Never Assessed': 'Unknown', 'Unknown': 'Unknown'
    })

    # --- Alcohol ---
    df['alcohol_use_clean'] = df['alcohol_use'].replace({
        'Yes': 'Current', 'Not Currently': 'Former',
        'No': 'Never', 'Never': 'Never', 'Not Asked': 'Unknown'
    })
    
    # --- IV drug use ---
    df['iv_drug_use_clean'] = df['iv_drug_use'].replace({'N': 0, 'Y': 1})
    
    # --- Sexually active ---
    df['sexually_active_clean'] = df['sexually_active'].replace({
        'Yes': 'Active', 'Not Currently': 'Inactive',
        'Never': 'Inactive', 'Not Asked': 'Unknown'
    })
    
    return df

# Step 1: clean social history
social_clean = clean_social_history(social_hist_feat)

In [31]:
def compute_social_features(social, patients):
    social = social.copy()

    # -------------------------------
    # 1) Attach patient timing info
    # -------------------------------
    patients_sub = patients[['pat_id', 'cp_label', 'cp_date',
                             'earliest_ap_date', 'last_ap_date']].copy()
    patients_sub['window_end'] = patients_sub['earliest_ap_date'] + pd.to_timedelta(365.25 * 10, unit='D')

    social = social.merge(patients_sub, on='pat_id', how='inner')

    # Ensure we have a timestamp column; adjust name if needed
    social['contact_date'] = pd.to_datetime(social['contact_date'], errors='coerce')

    # -------------------------------
    # 2) Apply time filtering rules
    # -------------------------------
    t = social['contact_date']

    # CP patients → keep only entries up to CP date
    mask_cp = (social['cp_label'] == 1) & (t < social['cp_date'])

    # Non-CP patients → keep only entries up to last AP date
    mask_censor = (social['cp_label'] == 0) & (t <= social['last_ap_date'])

    social = social[(mask_cp | mask_censor)].copy()

    # -------------------------------
    # 3) Map categorical to numeric
    # -------------------------------
    map_dict = {
        'tobacco_use_clean':      {'Never': 0, 'Former': 1, 'Current': 2, 'Unknown': np.nan},
        'smokeless_tobacco_use':  {'Never': 0, 'Former': 1, 'Current': 2, 'Unknown': np.nan},
        'alcohol_use_clean':      {'Never': 0, 'Former': 1, 'Current': 2, 'Unknown': np.nan},
        'sexually_active_clean':  {'Inactive': 0, 'Active': 1, 'Unknown': np.nan},
    }

    social_numeric = social.copy()
    for col, mapping in map_dict.items():
        if col in social_numeric.columns:
            social_numeric[col] = social_numeric[col].map(mapping)

    # -------------------------------
    # 4) Aggregate per patient
    # -------------------------------
    social_agg = social_numeric.groupby('pat_id').agg(
        max_tobacco=('tobacco_use_clean', 'max'),          # 0 Never, 1 Former, 2 Current
        max_smokeless=('smokeless_tobacco_use', 'max'),
        max_alcohol=('alcohol_use_clean', 'max'),
        ever_iv_drug=('iv_drug_use_clean', 'max'),         # assume already 0/1 or NaN
        ever_sexually_active=('sexually_active_clean', 'max')
    ).reset_index()

    # -------------------------------
    # 5) Include all patients
    # -------------------------------
    social_agg_full = patients[['pat_id']].merge(social_agg, on='pat_id', how='left')

    return social_agg_full


In [32]:
social_features = compute_social_features(social_clean, patients)

In [ ]:
social_features

# **Notes**

In [34]:
notes_feat = notes.copy()

In [ ]:
notes_feat.head()

In [36]:
notes_feat["note_specialty"].unique()

array(['Medicine, Gastroenterology', 'Pediatrics, Gastroenterology', nan],
      dtype=object)

In [37]:
notes_feat["note_type"].unique()

array(['Consults', 'Progress Notes', 'Discharge Narrative', 'H&P',
       'Brief Op Notes', 'Op Notes', nan], dtype=object)

In [38]:
notes_feat = notes_feat.copy()

# --- 1. Convert time columns ---
notes_feat['note_signed_time'] = pd.to_datetime(notes_feat['note_signed_time'], errors='coerce')

# --- 2. Attach patient timing info ---
patients_sub = patients[['pat_id', 'cp_label', 'cp_date',
                         'earliest_ap_date', 'last_ap_date']].copy()
patients_sub['window_end'] = patients_sub['earliest_ap_date'] + pd.to_timedelta(365.25 * 10, unit='D')

notes_feat = notes_feat.merge(patients_sub, on='pat_id', how='inner')

# --- 3. Apply time filtering rules using note_signed_time ---
t = notes_feat['note_signed_time']

# CP patients → keep notes up to CP date
mask_cp = (notes_feat['cp_label'] == 1) & (t < notes_feat['cp_date'])

# Non-CP patients → keep notes up to last AP date
mask_censor = (notes_feat['cp_label'] == 0) & (t <= notes_feat['last_ap_date'])

notes_feat = notes_feat[(mask_cp | mask_censor)].copy()

# --- 4. Sort notes by patient and time ---
notes_feat = notes_feat.sort_values(['pat_id', 'note_signed_time'])

# --- 5. Restrict to GI-related notes ---
notes_gi = notes_feat[
    notes_feat['note_specialty'].str.contains('Gastroenterology', na=False)
].copy()

# --- 6. Concatenate text per patient in chronological order ---
notes_concat = (
    notes_gi.groupby('pat_id', as_index=False)
    .agg(
        full_text_concat=('full_text', lambda x: list(x)),  # or ' '.join(x) if you want one long string
    )
)

# optional: include all patients, even with no notes
notes_concat_full = patients[['pat_id']].merge(notes_concat, on='pat_id', how='left')



In [ ]:
notes_concat_full[notes_concat_full["full_text_concat"].notna()]

# **Merge**

In [ ]:
patients.head()

In [ ]:
basic_info_feat.head()

In [42]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# 0) Base cohort from patients
#    (keep only the cohort columns you actually need)
# ---------------------------------------------------
preferred_base_cols = [
    'pat_id',
    'ap_date_set',
    'cp_date',
    'earliest_ap_date',
    'days_ap_to_cp',
    'ap_count',
    'cp_label',
    'time_to_cp_days',
    'last_ap_date',
]

# Only keep the ones that exist in patients
base_cols = [c for c in preferred_base_cols if c in patients.columns]

final_df = patients[base_cols].copy()

print("Base final_df shape (from patients):", final_df.shape)


# ---------------------------------------------------
# 1) Helper: drop duplicated cohort columns from features
# ---------------------------------------------------
def features_only(df, name="df"):
    df = df.copy()
    # Drop any columns that overlap with base cohort info, except pat_id
    overlap = [c for c in df.columns if c in base_cols and c != 'pat_id']
    if overlap:
        print(f"[{name}] dropping duplicate cohort columns:", overlap)
        df = df.drop(columns=overlap)
    return df


# ---------------------------------------------------
# 2) Clean each feature table so it only has pat_id + features
# ---------------------------------------------------
basic_info_feat_clean   = features_only(basic_info_feat,   "basic_info")
weight_height_clean     = features_only(weight_height,     "weight_height")
labs_mean_clean         = features_only(labs_mean,         "labs_mean")
med_features_clean      = features_only(med_features,      "med_features")
surgery_features_clean  = features_only(surgery_features,  "surgery_features")
social_features_clean   = features_only(social_features,   "social_features")
notes_concat_full_clean = features_only(notes_concat_full, "notes_concat_full")


# ---------------------------------------------------
# 3) Merge all features into final_df
#    (left join keeps all patients from `patients`)
# ---------------------------------------------------
final_df = final_df.merge(basic_info_feat_clean,   on='pat_id', how='left')
final_df = final_df.merge(weight_height_clean,     on='pat_id', how='left')
final_df = final_df.merge(labs_mean_clean,         on='pat_id', how='left')
final_df = final_df.merge(med_features_clean,      on='pat_id', how='left')
final_df = final_df.merge(surgery_features_clean,  on='pat_id', how='left')
final_df = final_df.merge(social_features_clean,   on='pat_id', how='left')
final_df = final_df.merge(notes_concat_full_clean, on='pat_id', how='left')

print("final_df shape after merges:", final_df.shape)


# ---------------------------------------------------
# 4) Final cleanup of specific columns
# ---------------------------------------------------
# Drop binary indicator if you only want count
if "ever_had_cholecystectomy" in final_df.columns:
    final_df = final_df.drop("ever_had_cholecystectomy", axis=1)

# num_cholecystectomies → 0 if NaN, cast to int
if "num_cholecystectomies" in final_df.columns:
    final_df["num_cholecystectomies"] = (
        final_df["num_cholecystectomies"].fillna(0).astype(int)
    )

# ever_taken_pancreatic_enzymes → 0/1 int
if "ever_taken_pancreatic_enzymes" in final_df.columns:
    final_df["ever_taken_pancreatic_enzymes"] = (
        final_df["ever_taken_pancreatic_enzymes"]
        .fillna(0)
        .astype(int)
    )
if "num_prescriptions" in final_df.columns:
    final_df["num_prescriptions"] = (
        final_df["num_prescriptions"]
        .fillna(0)
        .astype(int)
    )
print("Final columns:", final_df.columns.tolist())
print("Final shape:", final_df.shape)


Base final_df shape (from patients): (4904, 9)
[basic_info] dropping duplicate cohort columns: ['ap_date_set', 'cp_date', 'earliest_ap_date', 'days_ap_to_cp', 'ap_count', 'cp_label', 'time_to_cp_days', 'last_ap_date']
final_df shape after merges: (4904, 34)
Final columns: ['pat_id', 'ap_date_set', 'cp_date', 'earliest_ap_date', 'days_ap_to_cp', 'ap_count', 'cp_label', 'time_to_cp_days', 'last_ap_date', 'age', 'race', 'ethnicity', 'sex', 'alive', 'weight_kg', 'height_cm', 'AMYLASE', 'CREATININE', 'GLUCOSE', 'LIPASE', 'ever_taken_pancreatic_enzymes', 'num_prescriptions', 'first_med_start', 'last_med_end', 'num_cholecystectomies', 'first_surgery_date', 'last_surgery_date', 'max_tobacco', 'max_smokeless', 'max_alcohol', 'ever_iv_drug', 'ever_sexually_active', 'full_text_concat']
Final shape: (4904, 33)


In [ ]:
final_df

In [44]:
final_df.columns

Index(['pat_id', 'ap_date_set', 'cp_date', 'earliest_ap_date', 'days_ap_to_cp',
       'ap_count', 'cp_label', 'time_to_cp_days', 'last_ap_date', 'age',
       'race', 'ethnicity', 'sex', 'alive', 'weight_kg', 'height_cm',
       'AMYLASE', 'CREATININE', 'GLUCOSE', 'LIPASE',
       'ever_taken_pancreatic_enzymes', 'num_prescriptions', 'first_med_start',
       'last_med_end', 'num_cholecystectomies', 'first_surgery_date',
       'last_surgery_date', 'max_tobacco', 'max_smokeless', 'max_alcohol',
       'ever_iv_drug', 'ever_sexually_active', 'full_text_concat'],
      dtype='object')

In [45]:
final_df[(final_df['time_to_cp_days']<=3650) & (final_df['time_to_cp_days']>1825)]['time_to_cp_days'].describe()

count      93.000000
mean     2538.268817
std       467.736946
min      1837.000000
25%      2133.000000
50%      2533.000000
75%      2932.000000
max      3461.000000
Name: time_to_cp_days, dtype: float64

In [46]:
cols_to_check = final_df.loc[:,:] 
missing_info = cols_to_check.isna().sum().to_frame(name='missing_count')
missing_info['missing_ratio'] = missing_info['missing_count'] / len(final_df)
missing_info = missing_info.sort_values('missing_ratio', ascending=False)

print(missing_info)

                               missing_count  missing_ratio
last_med_end                            4697       0.957790
first_med_start                         4632       0.944535
last_surgery_date                       4330       0.882953
first_surgery_date                      4330       0.882953
cp_date                                 4219       0.860318
days_ap_to_cp                           4219       0.860318
time_to_cp_days                         4219       0.860318
AMYLASE                                 3199       0.652325
ever_sexually_active                    2785       0.567904
full_text_concat                        2639       0.538132
max_smokeless                            967       0.197186
LIPASE                                   793       0.161705
max_alcohol                              635       0.129486
max_tobacco                              433       0.088295
GLUCOSE                                  409       0.083401
CREATININE                              

In [47]:
final_df = final_df.drop(["last_med_end","first_med_start","last_surgery_date","first_surgery_date"],axis = 1)
final_df["num_prescriptions"] = final_df["num_prescriptions"].fillna(0).astype(int)


In [ ]:
final_df

In [ ]:
import pandas as pd
import numpy as np
df = final_df.copy()

# Compute helper columns
df['first_ap'] = df['ap_date_set'].apply(lambda x: min(x) if x else np.nan)
df['last_ap'] = df['ap_date_set'].apply(lambda x: max(x) if x else np.nan)

# Compute time_to_event
def compute_time_to_event(row):
    if pd.isna(row['cp_date']):
        return np.nan
    else:
        return (row['cp_date'] - row['first_ap']).days

df['time_to_event'] = df.apply(compute_time_to_event, axis=1)

# Compute time_of_observation
def compute_time_of_obs(row):
    if pd.notna(row['time_to_event']):
        return row['time_to_event']
    else:
        return (row['last_ap'] - row['first_ap']).days

df['time_of_observation'] = df.apply(compute_time_of_obs, axis=1)

df[['pat_id', 'time_to_event', 'time_of_observation']]


In [50]:
# Convert days to years
df['time_to_event_years'] = df['time_to_event'] / 365.25

# Define window labels
def categorize_time_window(t):
    if pd.isna(t):
        return np.nan
    elif t <= 5:
        return "0-5 years"
    elif t <= 10:
        return "5-10 years"
    else:
        return ">10 years"

df['time_window'] = df['time_to_event_years'].apply(categorize_time_window)


In [51]:
df['time_window'].value_counts()

time_window
0-5 years     565
5-10 years     93
>10 years      27
Name: count, dtype: int64

In [52]:
df = df.drop(['earliest_ap_date','time_to_cp_days'],axis = 1)



In [ ]:
df['cp_label'] = df['cp_label'].replace(3, 2)
df.iloc[0]

In [54]:
df.to_csv("output.csv", index=False)

In [55]:
df["cp_label"].value_counts()

cp_label
0    4219
1     685
Name: count, dtype: int64

In [56]:
sss = pd.read_csv("output.csv")

In [ ]:
sss[sss["time_window"] == ">10 years"]

In [58]:
sss.dtypes

pat_id                            object
ap_date_set                       object
cp_date                           object
days_ap_to_cp                    float64
ap_count                           int64
cp_label                           int64
last_ap_date                      object
age                              float64
race                              object
ethnicity                         object
sex                              float64
alive                              int64
weight_kg                        float64
height_cm                        float64
AMYLASE                          float64
CREATININE                       float64
GLUCOSE                          float64
LIPASE                           float64
ever_taken_pancreatic_enzymes      int64
num_prescriptions                  int64
num_cholecystectomies              int64
max_tobacco                      float64
max_smokeless                    float64
max_alcohol                      float64
ever_iv_drug    

In [59]:
import re
df = sss.copy()
def parse_ap_dates_string(x):
    if pd.isna(x) or x.strip() in ["{}", "[]"]:
        return []
    
    # Extract all date-like strings inside single quotes
    matches = re.findall(r"'([\d-]{10} [\d:]{8})'", x)
    if not matches:
        # Try to extract shorter date formats (e.g., '2021-12-01')
        matches = re.findall(r"'([\d-]{10})'", x)
    
    # Convert to datetime and sort
    datetimes = sorted(pd.to_datetime(matches, errors='coerce'))
    return datetimes

df['ap_date'] = df['ap_date_set'].apply(parse_ap_dates_string)

In [ ]:
df['ap_date'][0][1]